# Training a worm classifier

The following notebook closely follows the pytorch tutotrial here:
https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html

Before running on your own data, build your own dataset using the worm segmentation pipeline (detection.ipynb) and construct your own dataset using the folder structure in this example (project/example_train_classifier).

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
import os
from PIL import Image
from tempfile import TemporaryDirectory
from tqdm import tqdm

In [2]:
os.chdir(os.getcwd())

Define training functions

In [3]:
cudnn.benchmark = True
plt.ion()   # interactive mode
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def loadBag(data_dir):
    # Data augmentation and normalization for training
    # Just normalization for validation
    data_transforms = {
        'train': transforms.Compose([
            transforms.RandomResizedCrop(224),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    }
    
    image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x),
                                              data_transforms[x])
                      for x in ['train']}
    
    dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=4,
                                                 shuffle=True, num_workers=4)
                  for x in ['train']}
    
    dataset_sizes = {x: len(image_datasets[x]) for x in ['train']}
    class_names = image_datasets['train'].classes
    
    return dataloaders, class_names, dataset_sizes


#training
def train_model(model, criterion, optimizer, scheduler, dataloaders, dataset_sizes, num_epochs=25):
    since = time.time()
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data.
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # zero the parameter gradients
                optimizer.zero_grad()

                # forward
                # track history if only in train
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # deep copy the model
            best_acc = epoch_acc
    
    

        print()
    
    
    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')
    return model




# Here the size of each output sample is set to 2.
# Alternatively, it can be generalized to ``nn.Linear(num_ftrs, len(class_names))``.

def run_training(model_ft, num_ftrs, model, dataloaders, dataset_sizes, num_epochs=25):
    
    model_ft.fc = nn.Linear(num_ftrs, 2)
    
    model_ft = model_ft.to(device)
    
    criterion = nn.CrossEntropyLoss()
    
    # Observe that all parameters are being optimized
    optimizer_ft = optim.SGD(model_ft.parameters(), lr=0.001, momentum=0.9)
    
    # Decay LR by a factor of 0.1 every 7 epochs
    exp_lr_scheduler = lr_scheduler.StepLR(optimizer_ft, step_size=7, gamma=0.1)
    
    model_ft = train_model(model_ft, criterion, optimizer_ft, exp_lr_scheduler, dataloaders, dataset_sizes,
                           num_epochs=num_epochs)
    
    return model_ft



Define model parameters

In [4]:
model_ft = models.resnet18(weights='DEFAULT')
num_ftrs = model_ft.fc.in_features
data_dir = 'project/example_train_classifier/data_small/data_small'

Train and save model

In [6]:
dataloaders, class_names, dataset_sizes = loadBag(data_dir)
model = run_training(model_ft, num_ftrs, model_ft, dataloaders, dataset_sizes, num_epochs=25)
torch.save(model.state_dict(), 'project/example_train_classifier/classifier_small.pth')

Epoch 0/24
----------
train Loss: 0.8998 Acc: 0.5300

Epoch 1/24
----------
train Loss: 0.8156 Acc: 0.6150

Epoch 2/24
----------
train Loss: 0.8589 Acc: 0.6900

Epoch 3/24
----------
train Loss: 0.8400 Acc: 0.5750

Epoch 4/24
----------
train Loss: 0.7842 Acc: 0.5800

Epoch 5/24
----------
train Loss: 0.8000 Acc: 0.5650

Epoch 6/24
----------
train Loss: 0.6908 Acc: 0.6450

Epoch 7/24
----------
train Loss: 0.5894 Acc: 0.6900

Epoch 8/24
----------
train Loss: 0.6124 Acc: 0.6750

Epoch 9/24
----------
train Loss: 0.5533 Acc: 0.7350

Epoch 10/24
----------
train Loss: 0.5471 Acc: 0.6950

Epoch 11/24
----------
train Loss: 0.6043 Acc: 0.6550

Epoch 12/24
----------
train Loss: 0.5447 Acc: 0.7050

Epoch 13/24
----------
train Loss: 0.5600 Acc: 0.7350

Epoch 14/24
----------
train Loss: 0.5442 Acc: 0.7150

Epoch 15/24
----------
train Loss: 0.5478 Acc: 0.7500

Epoch 16/24
----------
train Loss: 0.5185 Acc: 0.7400

Epoch 17/24
----------
train Loss: 0.5181 Acc: 0.7350

Epoch 18/24
--------